# Week 88, Day 5: Code review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w088d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('sprint1.diff', 'w', encoding='utf-8') as f:
    f.write('diff --git a/src/waymark/storage.py b/src/waymark/storage.py\n--- a/src/waymark/storage.py\n+++ b/src/waymark/storage.py\n@@ -10,6 +10,18 @@ def load_network(conn):\n     return network\n \n+def find_routes(conn, text):\n+    rows = conn.execute(f"SELECT name FROM saved_route WHERE name LIKE \'%{text}%\'")\n+    return [r[0] for r in rows]\n+\n+def add_walks(conn, walks=[]):\n+    for walk in walks:\n+        conn.execute("INSERT INTO walk VALUES (?, ?)", (walk["day"], walk["km"]))\n+    return walks\n+\n+def walk_count(conn):\n+    return conn.execute("SELECT COUNT(*) FROM walk").fetchone()[0]\n+\n def save_network(conn, heights, paths):\n     with conn:\ndiff --git a/src/waymark/services.py b/src/waymark/services.py\n--- a/src/waymark/services.py\n+++ b/src/waymark/services.py\n@@ -3,8 +3,14 @@ from waymark.storage import load_network\n \n def plan(conn, start, end):\n-    route = load_network(conn).shortest(start, end)\n+    print("planning", start, end)\n+    try:\n+        route = load_network(conn).shortest(start, end)\n+    except:\n+        return None\n     return {"stops": route.stops, "km": round(route.km, 1)}\n+\n+def year_total(walks):\n+    if walks == None:\n+        return 0\n+    # TODO: count only this year\n+    return sum(w["km"] for w in walks)\ndiff --git a/tests/test_services.py b/tests/test_services.py\n--- a/tests/test_services.py\n+++ b/tests/test_services.py\n@@ -1,3 +1,6 @@\n from waymark.services import plan\n \n def test_plan_returns_route():\n     assert plan(CONN, "Mill", "Weir")["stops"] == ["Mill", "Weir"]\n+\n+def test_year_total_empty():\n+    assert year_total([]) == 0\n')

**Predict before you run:** what will this print?



In [ ]:
def add_walk(walk, log=[]):
    log.append(walk)
    return log

first = add_walk({"km": 5})
second = add_walk({"km": 8})
print(len(first), len(second), first is second)

> **Key idea.** Review in order: correctness, tests, design, security and data, readability; leave style to tools. Keep pull requests small, with a why and their tests. Label comments (blocking, suggestion, nit, question), be specific, suggest fixes, and talk about the code. End with a verdict; any blocking comment requests changes. Review your own pull requests a day later. Unified diff hunks (`@@ -old,len +new,len @@`) give line numbers: context and added lines advance the new count. Watch for SQL from strings, bare excepts, mutable defaults, leftover prints and TODOs, `== None`, N+1 queries and missing transactions or tests; automate what's mechanical.

### Exercise: Counting a diff

Write `diff_stats(diff_text)`: a dict from each changed file (the path after `+++ b/`) to `(added, removed)`: the number of lines starting with `+` or `-` in its hunks, not counting the `+++` and `---` header lines. Files appear in the order of the diff. Then `total_changed(stats)`: all added plus removed lines.

In [ ]:
def diff_stats(diff_text):
    ___

def total_changed(stats):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stats():\n    \"headers aren't changes\"\n    stats = diff_stats(open(\"sprint1.diff\").read())\n    assert stats == {\"src/waymark/storage.py\": (12, 0), \"src/waymark/services.py\": (11, 1), \"tests/test_services.py\": (3, 0)}\n    assert list(stats) == [\"src/waymark/storage.py\", \"src/waymark/services.py\", \"tests/test_services.py\"]\n    assert total_changed(stats) == 27")

### Exercise: Where each added line is

Write `added_lines(diff_text)`: a list of `(path, line_number, text)` for every added line, where `line_number` is its number in the **new** file, worked out from the hunk headers (`@@ -a,b +c,d @@`: the new file's lines start at `c`; context and added lines advance the count, removed lines don't), and `text` is the line without its `+`.

In [ ]:
import re

def added_lines(diff_text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lines():\n    \"hunk arithmetic\"\n    found = added_lines(open(\"sprint1.diff\").read())\n    assert found[0] == (\"src/waymark/storage.py\", 12, \"def find_routes(conn, text):\")\n    assert found[5] == (\"src/waymark/storage.py\", 17, \"    for walk in walks:\")\n    services = [f for f in found if f[0] == \"src/waymark/services.py\"]\n    assert services[0] == (\"src/waymark/services.py\", 5, '    print(\"planning\", start, end)')\n    assert services[-1] == (\"src/waymark/services.py\", 16, '    return sum(w[\"km\"] for w in walks)')\n    assert found[-1] == (\"tests/test_services.py\", 7, \"    assert year_total([]) == 0\")\n    assert len(found) == 26")

### Exercise: Labels and a verdict

Write `label(comment)`: the label at the start of a comment (`"blocking"`, `"suggestion"`, `"nit"` or `"question"`, before a colon, ignoring case), or `"unlabelled"`. Then `verdict(comments)`: `"changes requested"` if any comment is blocking, `"approved with comments"` if there are other comments, else `"approved"`; and `summary(comments)`: a dict from each label present to its count, in the order blocking, suggestion, question, nit, unlabelled.

In [ ]:
def label(comment):
    ___

def verdict(comments):
    ___

def summary(comments):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_comments():\n    \"how much each matters\"\n    comments = [\"blocking: SQL built with an f-string on line 13\", \"nit: use is None\", \"Question: why print here?\",\n                \"suggestion: rename year_total to km_this_year\", \"looks good\", \"NIT: trailing comma\"]\n    assert [label(c) for c in comments] == [\"blocking\", \"nit\", \"question\", \"suggestion\", \"unlabelled\", \"nit\"]\n    assert verdict(comments) == \"changes requested\" and verdict(comments[1:]) == \"approved with comments\" and verdict([]) == \"approved\"\n    assert summary(comments) == {\"blocking\": 1, \"suggestion\": 1, \"question\": 1, \"nit\": 2, \"unlabelled\": 1}\n    assert list(summary(comments)) == [\"blocking\", \"suggestion\", \"question\", \"nit\", \"unlabelled\"]")

### Exercise: One query, not one per route

The given `CountingConnection` wraps a SQLite connection and counts `execute` calls, and `routes_slow(conn)` loads every saved route with its stops using one query for the routes and one **per route** for its stops. Write `routes_fast(conn)` returning the same result (a dict from name to the list of stops in order, routes in name order) with **one** query, using a `JOIN` and grouping in Python.

In [ ]:
import sqlite3

class CountingConnection:
    def __init__(self, conn):
        self.conn, self.queries = conn, 0
    def execute(self, sql, params=()):
        self.queries += 1
        return self.conn.execute(sql, params)

def make_db(n_routes):
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE saved_route (id INTEGER PRIMARY KEY, name TEXT)")
    conn.execute("CREATE TABLE saved_stop (route_id INTEGER, position INTEGER, junction TEXT)")
    for i in range(n_routes):
        conn.execute("INSERT INTO saved_route VALUES (?, ?)", (i + 1, f"Route {i:03d}"))
        conn.executemany("INSERT INTO saved_stop VALUES (?, ?, ?)", [(i + 1, p, f"J{(i + p) % 7}") for p in range(3)])
    return conn

def routes_slow(conn):
    result = {}
    for route_id, name in conn.execute("SELECT id, name FROM saved_route ORDER BY name").fetchall():
        rows = conn.execute("SELECT junction FROM saved_stop WHERE route_id = ? ORDER BY position", (route_id,))
        result[name] = [r[0] for r in rows]
    return result

def routes_fast(conn):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_one_query():\n    \"same answer, one query\"\n    slow_conn, fast_conn = CountingConnection(make_db(50)), CountingConnection(make_db(50))\n    expected = routes_slow(slow_conn)\n    assert routes_fast(fast_conn) == expected and list(routes_fast(CountingConnection(make_db(50)))) == list(expected)\n    assert slow_conn.queries == 51 and fast_conn.queries == 1\n    assert routes_fast(CountingConnection(make_db(0))) == {}")

### Exercise: Is the pull request reviewable?

Write `pr_problems(stats, description, limit=400)` for diff stats like `diff_stats`'s and a pull request description: a list of, in order, `"too big: N changed lines"` (more than `limit`), `"no tests changed"` (no changed file under `tests/`), `"no story linked"` (no story id like `S3` in the description) and `"no why"` (the description has fewer than 10 words).

In [ ]:
import re

def pr_problems(stats, description, limit=400):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pr():\n    \"small, tested, explained\"\n    stats = {\"src/waymark/storage.py\": (12, 0), \"src/waymark/services.py\": (11, 1), \"tests/test_services.py\": (3, 0)}\n    good = \"S3 and S5: search saved routes by name, and total a year's walks, because walkers asked for both.\"\n    assert pr_problems(stats, good) == []\n    assert pr_problems({\"src/a.py\": (380, 30)}, \"stuff\") == [\"too big: 410 changed lines\", \"no tests changed\", \"no story linked\", \"no why\"]\n    assert pr_problems(stats, \"Adds search and totals for walkers who keep a log of their walks\") == [\"no story linked\"]")

### Bug hunt: A shared default

The default log is shared between calls, so the second walker's log already holds the first walker's walk: it prints `1 2`. Use `None` as the default and make a new list inside, so it prints `1 1`.

In [ ]:
def start_log(first_walk, log=[]):
    log.append(first_walk)
    return log

ada = start_log({"km": 5})
bo = start_log({"km": 8})
print(len(ada), len(bo))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"None, then a new list\"\n    expect_output(\"1 1\")")

### Bug hunt: An error that disappeared

The bare `except` turns a typing mistake inside the function (`stop` for `stops`) into a quiet `None`, so the program prints `None` and the bug hides. Catch only the error you expect (`KeyError` for an unknown route) so the real mistake shows up, and fix it, so it prints `['Mill', 'Weir']`.

In [ ]:
ROUTES = {"Canal": {"stops": ["Mill", "Weir"]}}

def stops_of(name):
    try:
        return ROUTES[name]["stop"]
    except:
        return None

print(stops_of("Canal"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"catch what you expect\"\n    expect_output(\"['Mill', 'Weir']\")\n    assert \"except:\" not in source and \"except Exception\" not in source, \"catch a specific exception\"")

### Puzzle: Reviewing your own pull request

Arrange a routine for reviewing your own pull request. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the new file's first line number in the hunk "@@ -4,7 +9,8 @@"
q = ___   # the label for a comment about a missing transaction around two related writes
r = ___   # queries run by an N+1 loader for 20 routes
s = ___   # the verdict for a review with two nits and one question
t = ___   # True or False: a review should argue about quote style

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 9 and q == \"blocking\" and r == 21, \"p, q and r\"\n    assert s == \"approved with comments\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an automatic reviewer

Write `review(diff_text)` using the given `added_lines`. For each **added** line, add a comment for each rule it breaks, checked in this order: a bare `except:` gives `"blocking: bare except hides errors"`; an f-string (`f"` or `f'`) containing `SELECT`, `INSERT`, `UPDATE` or `DELETE` gives `"blocking: SQL built with an f-string"`; a `def` with a default of `[]`, `{}` or `set()` gives `"blocking: mutable default argument"`; a call to `print(` gives `"nit: leftover print"`; `== None` or `!= None` gives `"nit: compare with None using is"`; `TODO` gives `"question: TODO without an issue"`. Return `(comments, verdict)`: `comments` as `"PATH:LINE: COMMENT"` strings, sorted by path, then line number, then rule order; `verdict` as in the exercises (`"changes requested"` if any is blocking, `"approved with comments"` if any comment, else `"approved"`).

In [ ]:
import re

def added_lines(diff_text):
    found, path, number = [], None, 0
    for line in diff_text.splitlines():
        if line.startswith("+++ b/"):
            path = line[6:]
        elif line.startswith(("+++", "---", "diff ")):
            continue
        elif line.startswith("@@"):
            number = int(re.match(r"@@ -\d+(?:,\d+)? \+(\d+)", line).group(1))
        elif line.startswith("+"):
            found.append((path, number, line[1:]))
            number += 1
        elif not line.startswith("-"):
            number += 1
    return found

def review(diff_text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_review():\n    \"everything a machine can find\"\n    comments, verdict = review(open(\"sprint1.diff\").read())\n    assert comments == [\n        \"src/waymark/services.py:5: nit: leftover print\",\n        \"src/waymark/services.py:8: blocking: bare except hides errors\",\n        \"src/waymark/services.py:13: nit: compare with None using is\",\n        \"src/waymark/services.py:15: question: TODO without an issue\",\n        \"src/waymark/storage.py:13: blocking: SQL built with an f-string\",\n        \"src/waymark/storage.py:16: blocking: mutable default argument\",\n    ]\n    assert verdict == \"changes requested\"\n\ndef test_clean():\n    \"nothing to say\"\n    clean = \"+++ b/src/x.py\\n@@ -1,1 +1,2 @@\\n x = 1\\n+y = x is None\\n\"\n    assert review(clean) == ([], \"approved\")\n    nits = \"+++ b/a.py\\n@@ -0,0 +1,2 @@\\n+print(f'SELECT is a word')\\n+def f(x=set()): pass\\n\"\n    assert review(nits) == ([\"a.py:1: blocking: SQL built with an f-string\", \"a.py:1: nit: leftover print\",\n                             \"a.py:2: blocking: mutable default argument\"], \"changes requested\")")

> **What the machine missed.** The reviewer found six problems in `sprint1.diff`, and missed at least three that matter more. `add_walks` writes several rows without a transaction. `plan` now returns `None` for an unknown place, so the API that called it will crash on `None["stops"]` instead of answering 404. And `test_year_total_empty` uses `year_total` without importing it, so the test fails, and the TODO hides a real bug: the total isn't for the year at all. None of those match a pattern; all of them need someone who knows what the code is for. That's the division of labour: machines for the mechanical, people for the meaning.

## Warm-up

The default list is made **once**, when the function is defined, and shared by every call that doesn't pass its own: the second walk lands in the first call's log. It's the kind of bug that passes a quick test and fails in production, and exactly what reviews are for. Open `sprint1.diff` above: it's a pull request from this sprint, and it has at least seven problems. Read it as a reviewer before the lesson does.

## What to look for

Review in order of importance, so the time goes where it matters:

1. **Correctness**: does it do what the story says, including the unhappy paths? Off-by-one errors, missing cases, wrong conditions, swallowed errors.
2. **Tests**: is the new behaviour tested, including failures? Would the tests catch a plausible bug?
3. **Design**: is each piece in the right layer (Week 87's dependency rule)? Is anything duplicated that already exists?
4. **Security and data**: user input in SQL, paths or HTML; secrets; transactions around related changes.
5. **Readability**: names that say what things are, functions that do one thing, comments that say why.

Style (spacing, quotes, import order) isn't on the list: leave it to the formatter and linter in CI, so reviews never argue about it.

Keep changes **small**: under about 400 changed lines, a reviewer's attention holds; above it, reviews become rubber stamps. A good pull request says **why** in its description, links the story, and includes its tests.

## Writing comments

A review comment should be:

- **Labelled** with how much it matters: `blocking:` (must change before merging), `suggestion:` (consider it), `nit:` (trivial; fine to ignore), `question:` (I don't understand: explain, maybe in the code).
- **Specific**: the line, the problem, and why it matters; ideally a suggested fix.
- **About the code, not the person**: "this query builds SQL from user text, so a name containing `'` breaks it", not "you forgot about injection again".

The review ends with a **verdict**: approve, approve with comments, or request changes (any blocking comment). For your own pull requests, read the diff the next day, write the comments to yourself in the pull request, and fix the blocking ones before merging. It feels odd, and it catches a surprising amount.

## Reading a diff

A **unified diff** (what `git diff` and pull requests show) lists changed files, each with **hunks**:

```
@@ -10,6 +10,18 @@ def load_network(conn):
```

means "the old file's lines 10 to 15 (6 lines) became the new file's lines 10 to 27 (18 lines)". Inside a hunk, lines starting with a space are unchanged context, `-` lines were removed and `+` lines added. Counting from the hunk's new start, context and added lines advance the new line number; removed lines don't. Tools that comment on pull requests use exactly this to place a comment on the right line.

## Common findings

The findings that come up again and again in Python reviews, most of them in `sprint1.diff`:

- **SQL built from strings** (an f-string with user text): injection and broken quotes. Use parameters (Week 79).
- **Bare `except:`** or `except Exception: return None`: hides real bugs and turns errors into silent wrong answers. Catch the specific exception you expect, and let the rest propagate.
- **Mutable default arguments** (`walks=[]`, `options={}`): shared between calls. Use `None` and create the list inside.
- **Leftover `print`** debugging, and `TODO`s with no issue: use logging (Week 91), and turn TODOs into backlog items.
- **`== None`**: use `is None`.
- **N+1 queries**: loading a list with one query, then running one more query **per item** (for each saved route, a query for its stops). Fine for 3 routes, painfully slow for 3,000. Fetch everything in one query with a `JOIN` and group in Python.
- **Missing transactions** around related writes, and **missing tests** for the new error paths.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Review the sprint's pull requests** (your own, from the start of the week) with today's order: correctness, tests, design, security, readability. Write labelled comments in the pull requests.
2. **Run `review`** over `git diff main...` for your branch, and add its rules (or a linter that does the same, such as ruff with the `B` rules) to CI.
3. **Fix every blocking comment**, each in its own small pull request.
4. **Swap reviews** with someone else on the course if you can: one pull request each, with the same checklist.

## Before you go

Count your blocking comments by kind. The most common kind is a habit to change, and maybe a rule to automate. Tomorrow is the **sprint review**: showing what was built, measuring the sprint, and planning the next one.